<a href="https://colab.research.google.com/github/ntomben97/NYC-Yellow-Taxi-Big-Data-Project/blob/main/Cleaning__Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pyspark installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install pyspark -q

Mounted at /content/drive


Creating Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("NYC Taxi Analysis") \
    .getOrCreate()

Loading Data

In [ ]:
DATA_PATH = "/content/drive/MyDrive/MIT805/Group Project/src"

df = spark.read.parquet(f"{DATA_PATH}/*.parquet")

df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2014-01-01 00:02:00|  2014-01-01 00:04:00|              6|          0.0|         1|              NULL|         146|         146|           1|        3.5|  0.5|    0.5|      0.0

In [ ]:
total_rows = df.count()
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(df.columns)}")

Total rows: 986,000,101
Total columns: 19


Column Name Discepancies

Using the Latest file as reference

In [ ]:
files = sorted(Path(DATA_PATH).glob("*.parquet"))

print(f"Number of files: {len(files)}")

reference_file = Path(DATA_PATH) / "yellow_tripdata_2026-05.parquet"

reference_df = spark.read.parquet(str(reference_file))

print("Reference file:")
print(reference_file.name)

reference_df.printSchema()

Number of files: 149
Reference file:
yellow_tripdata_2026-05.parquet
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [ ]:
reference_schema = {
    field.name.lower(): field.dataType.simpleString()
    for field in reference_df.schema.fields
}

print(reference_schema)

{'vendorid': 'int', 'tpep_pickup_datetime': 'timestamp_ntz', 'tpep_dropoff_datetime': 'timestamp_ntz', 'passenger_count': 'bigint', 'trip_distance': 'double', 'ratecodeid': 'bigint', 'store_and_fwd_flag': 'string', 'pulocationid': 'int', 'dolocationid': 'int', 'payment_type': 'bigint', 'fare_amount': 'double', 'extra': 'double', 'mta_tax': 'double', 'tip_amount': 'double', 'tolls_amount': 'double', 'improvement_surcharge': 'double', 'total_amount': 'double', 'congestion_surcharge': 'double', 'airport_fee': 'double', 'cbd_congestion_fee': 'double'}


In [ ]:
schema_differences = []

for file in files:

    current_df = spark.read.parquet(str(file))

    current_schema = {
      field.name.lower(): field.dataType.simpleString()
      for field in current_df.schema.fields
  }

    all_columns = set(reference_schema) | set(current_schema)

    for column in sorted(all_columns):

        reference_type = reference_schema.get(column, "MISSING")
        current_type = current_schema.get(column, "MISSING")

        if reference_type != current_type:

            schema_differences.append({
                "file": file.name,
                "column": column,
                "reference_type": reference_type,
                "actual_type": current_type
            })

In [ ]:
differences_pd = pd.DataFrame(schema_differences)

differences_pd

,file,column,reference_type,actual_type
0,yellow_tripdata_2014-01.parquet,airport_fee,double,int
1,yellow_tripdata_2014-01.parquet,cbd_congestion_fee,double,MISSING
2,yellow_tripdata_2014-01.parquet,dolocationid,int,bigint
3,yellow_tripdata_2014-01.parquet,pulocationid,int,bigint
4,yellow_tripdata_2014-01.parquet,vendorid,int,bigint
...,...,...,...,...
699,yellow_tripdata_2024-08.parquet,cbd_congestion_fee,double,MISSING
700,yellow_tripdata_2024-09.parquet,cbd_congestion_fee,double,MISSING
701,yellow_tripdata_2024-10.parquet,cbd_congestion_fee,double,MISSING
702,yellow_tripdata_2024-11.parquet,cbd_congestion_fee,double,MISSING


In [ ]:
differences_pd.groupby(
    ["column", "reference_type", "actual_type"]
).size().reset_index(name="number_of_files").sort_values(
    "number_of_files",
    ascending=False
)

,column,reference_type,actual_type,number_of_files
1,cbd_congestion_fee,double,MISSING,132
8,vendorid,int,bigint,109
3,dolocationid,int,bigint,109
6,pulocationid,int,bigint,109
0,airport_fee,double,int,79
5,passenger_count,bigint,double,55
2,congestion_surcharge,double,int,55
7,ratecodeid,bigint,double,55
4,improvement_surcharge,double,int,1


In [ ]:
target_types = {
    "vendorid": "bigint",
    "pulocationid": "bigint",
    "dolocationid": "bigint",
    "airport_fee": "double",
    "passenger_count": "double",
    "congestion_surcharge": "double",
    "ratecodeid": "bigint",
    "improvement_surcharge": "double",
    "cbd_congestion_fee": "double"
}

In [ ]:
standardised_dfs = []

for file in files:

    temp_df = spark.read.parquet(str(file))

    # Make column names lowercase
    for column in temp_df.columns:
        temp_df = temp_df.withColumnRenamed(
            column,
            column.lower()
        )

    # Standardise the required columns
    for column, target_type in target_types.items():

        if column in temp_df.columns:

            temp_df = temp_df.withColumn(
                column,
                F.col(column).cast(target_type)
            )

        else:

            # Column did not exist in this historical file
            temp_df = temp_df.withColumn(
                column,
                F.lit(None).cast(target_type)
            )

    standardised_dfs.append(temp_df)

In [ ]:
df_standardised = standardised_dfs[0]

for temp_df in standardised_dfs[1:]:
    df_standardised = df_standardised.unionByName(
        temp_df,
        allowMissingColumns=True
    )

df_standardised.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecodeid|store_and_fwd_flag|pulocationid|dolocationid|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|         1|              NULL|         146|    

In [ ]:
df_standardised.printSchema()

root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: long (nullable = true)
 |-- dolocationid: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Dealing with Null Values

In [ ]:
total_rows = df_standardised.count()

null_results = []

for column in df_standardised.columns:

    print(f"Checking: {column}")

    null_count = (
        df_standardised
        .select(
            F.sum(
                F.when(F.col(column).isNull(), 1)
                .otherwise(0)
            ).alias("null_count")
        )
        .collect()[0]["null_count"]
    )

    null_results.append({
        "Column": column,
        "Null_Count": null_count,
        "Null_Percentage": (null_count / total_rows) * 100
    })

    print(
        f"  Nulls: {null_count:,} "
        f"({(null_count / total_rows) * 100:.2f}%)"
    )

Checking: vendorid
  Nulls: 0 (0.00%)
Checking: tpep_pickup_datetime
  Nulls: 0 (0.00%)
Checking: tpep_dropoff_datetime
  Nulls: 0 (0.00%)
Checking: passenger_count
  Nulls: 25,984,554 (2.64%)
Checking: trip_distance
  Nulls: 0 (0.00%)
Checking: ratecodeid
  Nulls: 25,984,554 (2.64%)
Checking: store_and_fwd_flag
  Nulls: 76,104,792 (7.72%)
Checking: pulocationid
  Nulls: 0 (0.00%)
Checking: dolocationid
  Nulls: 0 (0.00%)
Checking: payment_type
  Nulls: 0 (0.00%)
Checking: fare_amount
  Nulls: 0 (0.00%)
Checking: extra
  Nulls: 0 (0.00%)
Checking: mta_tax
  Nulls: 0 (0.00%)
Checking: tip_amount
  Nulls: 0 (0.00%)
Checking: tolls_amount
  Nulls: 0 (0.00%)
Checking: improvement_surcharge
  Nulls: 53,750,503 (5.45%)
Checking: total_amount
  Nulls: 0 (0.00%)
Checking: congestion_surcharge
  Nulls: 689,772,615 (69.96%)
Checking: airport_fee
  Nulls: 797,072,315 (80.84%)
Checking: cbd_congestion_fee
  Nulls: 918,278,217 (93.13%)


In [ ]:
# ============================================================
# 1. Find the most common passenger_count
# ============================================================

passenger_mode = (
    df_standardised
    .filter(F.col("passenger_count").isNotNull())
    .groupBy("passenger_count")
    .count()
    .orderBy(F.desc("count"))
    .first()["passenger_count"]
)

print(f"Passenger count mode: {passenger_mode}")


# ============================================================
# 2. Replace NULL values
# ============================================================

df_cleaned = (
    df_standardised

    # Passenger count → mode
    .withColumn(
        "passenger_count",
        F.coalesce(
            F.col("passenger_count"),
            F.lit(float(passenger_mode))
        )
    )

    # Rate code → 99 = Unknown
    .withColumn(
        "ratecodeid",
        F.coalesce(
            F.col("ratecodeid"),
            F.lit(99).cast("bigint")
        )
    )

    # Store and forward flag → Unknown
    .withColumn(
        "store_and_fwd_flag",
        F.coalesce(
            F.col("store_and_fwd_flag"),
            F.lit("Unknown")
        )
    )

    # Improvement surcharge → 0
    .withColumn(
        "improvement_surcharge",
        F.coalesce(
            F.col("improvement_surcharge"),
            F.lit(0.0)
        )
    )

    # Congestion surcharge → 0
    .withColumn(
        "congestion_surcharge",
        F.coalesce(
            F.col("congestion_surcharge"),
            F.lit(0.0)
        )
    )

    # Airport fee → 0
    .withColumn(
        "airport_fee",
        F.coalesce(
            F.col("airport_fee"),
            F.lit(0.0)
        )
    )

    # CBD congestion fee → 0
    .withColumn(
        "cbd_congestion_fee",
        F.coalesce(
            F.col("cbd_congestion_fee"),
            F.lit(0.0)
        )
    )
)

Passenger count mode: 1.0


In [ ]:
cleaned_columns = [
    "passenger_count",
    "ratecodeid",
    "store_and_fwd_flag",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee"
]

null_check = df_cleaned.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in cleaned_columns
])

null_check.show(vertical=True)

-RECORD 0--------------------
 passenger_count       | 0   
 ratecodeid            | 0   
 store_and_fwd_flag    | 0   
 improvement_surcharge | 0   
 congestion_surcharge  | 0   
 airport_fee           | 0   
 cbd_congestion_fee    | 0   



In [ ]:
df = df_cleaned

# ============================================================
# 1. DEFINE VALIDATION CONDITIONS
# ============================================================

checks = {

    # VendorID must be 1, 2, 6 or 7
    "VendorID": ~F.col("VendorID").isin(1, 2, 6, 7),

    # Pickup date: January 2014 through May 2025
    "tpep_pickup_datetime": (
        (F.col("tpep_pickup_datetime") < F.lit("2014-01-01")) |
        (F.col("tpep_pickup_datetime") >= F.lit("2025-06-01"))
    ),

    # Dropoff date: January 2014 through May 2025
    "tpep_dropoff_datetime": (
        (F.col("tpep_dropoff_datetime") < F.lit("2014-01-01")) |
        (F.col("tpep_dropoff_datetime") >= F.lit("2025-06-01"))
    ),

    # Passenger count must not be negative
    "passenger_count": F.col("passenger_count") < 0,

    # Trip distance must not be negative
    "trip_distance": F.col("trip_distance") < 0,

    # Store and forward flag must be Y, N or UNKNOWN
    "store_and_fwd_flag": (
        ~F.upper(F.col("store_and_fwd_flag")).isin("Y", "N", "UNKNOWN")
    ),

    # Payment type must be between 0 and 6
    "payment_type": (
        (F.col("payment_type") < 0) |
        (F.col("payment_type") > 6)
    )
}


# ============================================================
# 2. AMOUNT COLUMNS
# ============================================================

amount_columns = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

# Only use amount columns that exist
amount_columns = [
    c for c in amount_columns
    if c in df.columns
]

for column in amount_columns:
    checks[column] = F.col(column) < 0


# ============================================================
# 3. COUNT INVALID VALUES
# ============================================================

total_rows = df.count()

results = []

for column_name, condition in checks.items():

    invalid_count = df.filter(condition).count()

    valid_count = total_rows - invalid_count

    invalid_percentage = (
        (invalid_count / total_rows) * 100
        if total_rows > 0 else 0
    )

    results.append((
        column_name,
        total_rows,
        valid_count,
        invalid_count,
        invalid_percentage
    ))


# ============================================================
# 4. CREATE VALIDATION REPORT
# ============================================================

validation_report = spark.createDataFrame(
    results,
    [
        "Column",
        "Total_Rows",
        "Valid_Rows",
        "Invalid_Rows",
        "Invalid_Percentage"
    ]
)

validation_report.orderBy(
    F.desc("Invalid_Rows")
).show(
    len(results),
    truncate=False
)

+---------------------+----------+----------+------------+---------------------+
|Column               |Total_Rows|Valid_Rows|Invalid_Rows|Invalid_Percentage   |
+---------------------+----------+----------+------------+---------------------+
|tpep_dropoff_datetime|986000101 |938033133 |47966968    |4.8648035584734695   |
|tpep_pickup_datetime |986000101 |938035958 |47964143    |4.864517047346631    |
|fare_amount          |986000101 |981018827 |4981274     |0.5052001510900453   |
|total_amount         |986000101 |983014300 |2985801     |0.30281954301747077  |
|improvement_surcharge|986000101 |983223251 |2776850     |0.28162776019837343  |
|mta_tax              |986000101 |983296504 |2703597     |0.27419845061456033  |
|congestion_surcharge |986000101 |983961525 |2038576     |0.20675210863898277  |
|extra                |986000101 |984606584 |1393517     |0.14133031006657068  |
|VendorID             |986000101 |985229844 |770257      |0.0781193631946697   |
|airport_fee          |98600

In [ ]:

validation_conditions = {
    "RatecodeID": ~F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99),
    "PULocationID": F.col("PULocationID") < 0,
    "DOLocationID": F.col("DOLocationID") < 0
}

results = []

for column_name, condition in validation_conditions.items():

    invalid_count = (
        df_cleaned
        .filter(condition)
        .count()
    )

    invalid_percentage = (invalid_count / total_rows) * 100

    results.append((
        column_name,
        total_rows,
        invalid_count,
        invalid_percentage
    ))

validation_df = spark.createDataFrame(
    results,
    ["Column", "Total_Rows", "Invalid_Rows", "Invalid_Percentage"]
)

validation_df.show(truncate=False)

+------------+----------+------------+------------------+
|Column      |Total_Rows|Invalid_Rows|Invalid_Percentage|
+------------+----------+------------+------------------+
|RatecodeID  |986000101 |0           |0.0               |
|PULocationID|986000101 |0           |0.0               |
|DOLocationID|986000101 |0           |0.0               |
+------------+----------+------------+------------------+



Looking into tpep_dropoff_datetime and tpep_pickup_datetime

In [ ]:
file_path = "/content/drive/MyDrive/MIT805/Group Project/src/yellow_tripdata_2022-06.parquet"

df_2022_06 = spark.read.parquet(file_path)

df_2022_06.filter(
    F.col("tpep_pickup_datetime") < F.lit("2014-01-01")
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "vendorid",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
).show(50, truncate=False)

+--------------------+---------------------+--------+---------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|vendorid|passenger_count|trip_distance|fare_amount|total_amount|
+--------------------+---------------------+--------+---------------+-------------+-----------+------------+
|2001-08-23 05:34:45 |2001-08-23 05:57:11  |2       |1.0            |7.08         |22.5       |24.55       |
|2002-10-21 00:13:16 |2002-10-21 00:24:37  |2       |1.0            |2.78         |11.0       |17.76       |
|2002-10-21 00:25:44 |2022-06-06 00:40:18  |2       |1.0            |3.01         |13.0       |20.16       |
|2002-10-21 00:27:54 |2002-10-21 00:44:31  |2       |2.0            |3.27         |13.5       |17.3        |
|2002-10-21 00:50:00 |2002-10-21 00:50:06  |2       |3.0            |0.0          |66.0       |66.3        |
|2002-10-21 01:38:48 |2022-06-06 01:49:22  |2       |1.0            |4.29         |14.5       |20.8        |
|2002-10-21 05:50:5

In [ ]:
# ---------------------------------------------------------
# Define the intended analysis period
# January 2014 through May 2026
# ---------------------------------------------------------
START_DATE = "2014-01-01"
END_DATE = "2026-06-01"   # exclusive, so May 2026 is included

# ---------------------------------------------------------
# Count records that fall outside the intended period
# ---------------------------------------------------------
out_of_range_count = (
    df_cleaned
    .filter(
        (F.col("tpep_pickup_datetime") < F.lit(START_DATE)) |
        (F.col("tpep_pickup_datetime") >= F.lit(END_DATE))
    )
    .count()
)

print(f"Records outside analysis period: {out_of_range_count:,}")

# ---------------------------------------------------------
# Create filtered dataframe
# ---------------------------------------------------------
df_filtered = (
    df_cleaned
    .filter(
        (F.col("tpep_pickup_datetime") >= F.lit(START_DATE)) &
        (F.col("tpep_pickup_datetime") < F.lit(END_DATE))
    )
)

# ---------------------------------------------------------
# Check the resulting date range
# ---------------------------------------------------------
df_filtered.select(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
).show(truncate=False)

# ---------------------------------------------------------
# Check row count after filtering
# ---------------------------------------------------------
filtered_count = df_filtered.count()

print(f"Rows after filtering: {filtered_count:,}")

Records outside analysis period: 2,712
+-------------------+-------------------+
|earliest_pickup    |latest_pickup      |
+-------------------+-------------------+
|2014-01-01 00:00:00|2026-05-31 23:59:59|
+-------------------+-------------------+

Rows after filtering: 985,997,389


Looking into negatives amounts

In [ ]:
df_filtered.filter(
    F.col("fare_amount") < 0
).select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "payment_type",
    "congestion_surcharge",
    "airport_fee"
).show(50, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+-----------+------+-------+----------+------------+---------------------+------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|extra |mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+-----------+------+-------+----------+------------+---------------------+------------+------------+--------------------+-----------+
|1       |2014-03-31 13:20:26 |2014-03-31 13:54:43  |1.0            |15.3         |-612.42    |628.37|30.05  |9.2       |0.0         |0.0                  |55.2        |1           |0.0                 |0.0        |
|2       |2014-08-01 00:14:19 |2014-08-01 00:17:33  |4.0            |0.39         |-4.0       |-0.5  |-0.5   |0.0       |0.0         |0.

Checking the VendorId

In [ ]:
(
    df_filtered
    .groupBy("VendorID")
    .count()
    .orderBy("VendorID")
    .show(truncate=False)
)

+--------+---------+
|VendorID|count    |
+--------+---------+
|1       |397538641|
|2       |586617738|
|3       |10297    |
|4       |758881   |
|5       |1079     |
|6       |299866   |
|7       |770887   |
+--------+---------+



Checking the negative trip distance

In [ ]:
(
    df_filtered
    .filter(F.col("trip_distance") < 0)
    .select(
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "payment_type"
    )
    .show(50, truncate=False)
)

+--------+--------------------+---------------------+---------------+-------------+-----------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|total_amount|payment_type|
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+------------+
|1       |2014-10-27 06:33:05 |2014-10-27 07:32:35  |1.0            |-3446729.6   |2.5        |13.16       |3           |
|1       |2015-02-05 07:02:26 |2015-02-05 07:02:29  |1.0            |-4.08401244E7|2.5        |3.8         |3           |
|1       |2015-02-07 15:56:13 |2015-02-07 15:57:03  |3.0            |-4.00955322E7|0.0        |1.3         |1           |
|1       |2015-04-24 08:27:04 |2015-04-24 08:29:40  |1.0            |-186318.4    |2.5        |3.3         |2           |
|2       |2019-11-20 16:51:31 |2019-11-20 17:51:56  |1.0            |-11.45       |39.59      |39.59       |2           |
|2       |2019-11-20 16:

In [ ]:
(
    df_standardised
    .filter(F.col("trip_distance") < 0)
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .groupBy("year")
    .count()
    .orderBy("year")
    .show()
)

+----+-----+
|year|count|
+----+-----+
|2014|    1|
|2015|    3|
|2019| 9102|
|2020| 2338|
+----+-----+



In [ ]:
df_filtered_2 = df_filtered.filter(F.col("trip_distance") >= 0)

print("Remaining rows with negative distance:")
df_filtered_2.filter(F.col("trip_distance") < 0).count()

Remaining rows with negative distance:


0

In [ ]:
df_filtered_2.write \
    .mode("overwrite") \
    .parquet("/content/drive/MyDrive/MIT805/Group Project/ProcessedData/CleanedData")